In [1]:
import sys
sys.path.append('..')

import os
os.environ["KERAS_BACKEND"] = "torch"

import time
import keras
import numpy as np
# Necesario para TextVectorization y tf.data.
import tensorflow as tf
from models.training import compile_model, get_callbacks
from config.settings import Settings
from features.embeddings import load_gensim_embeddings
from datasets.dataset import create_dataset
from features.vectorizer import build_vectorizer
from datasets.loader import load_splits, save_json
from models.siamese_lstm import SiameseLSTM
from datasets.paths import ProjectPaths


In [2]:
print(keras.config.backend())


torch


In [3]:
SEED = 42
np.random.seed(SEED)


In [4]:
settings = Settings()
print(settings)


embed_dim=400 batch_size=64 mlp_dropout=0.4 lstm_dropout=0.3 pooling='mean' similarity='manhattan' hidden_dim=64 bidirectional=False mlp_layers=[16] concat_features=['diff'] epochs=20 augmented_data=False siamese_name='lstm_mean_manhattan_noaug'


In [5]:
paths = ProjectPaths(siamese_name=settings.siamese_name)


In [6]:
if settings.augmented_data:
	max_len = 27
	train_dir = paths.augmented_dir
else:
	max_len = 26
	train_dir = paths.processed_dir


In [7]:
splits = {
	"train": train_dir,
	"dev": paths.processed_dir,
    "test": paths.processed_dir
}

datasets = load_splits(splits)

train_df = datasets["train"]
dev_df = datasets["dev"]
test_df = datasets["test"]


In [8]:
print("Train length:", len(train_df))
print("Dev length:", len(dev_df))


Train length: 5741
Dev length: 1497


In [9]:
all_sentences = list(train_df["sentence1"]) + list(train_df["sentence2"])

vectorizer = build_vectorizer(all_sentences, max_len)

vocab = vectorizer.get_vocabulary()
word2idx = {word: idx for idx, word in enumerate(vocab)}
print(f"Vocabulary size: {len(vocab)}")

vectorizer_model = keras.Sequential([vectorizer])
vectorizer_model.save(paths.vectorizer_path)


Vocabulary size: 13698


c:\Users\malos\Documents\GitHub\JustShare\server\.venv\Lib\site-packages\keras\src\saving\saving_api.py:107: UserWarning: You are saving a model that has not yet been built. It might not contain any weights yet. Consider building the model first by calling it on some data.
  return saving_lib.save_model(model, filepath)


In [10]:
embedding_matrix = load_gensim_embeddings(paths.word2vec_path, word2idx, settings.embed_dim)

np.save(paths.embedding_path, embedding_matrix)


Found 13300/13698 words


In [11]:
train_dataset = create_dataset(train_df, vectorizer, settings.batch_size, shuffle=True)
dev_dataset = create_dataset(dev_df, vectorizer, settings.batch_size)


In [12]:
for (sent1, sent2), y in train_dataset.take(1):
	print("sent1:", sent1.shape)
	print("sent2:", sent2.shape)
	print("y:", y.shape)


sent1: (64, 26)
sent2: (64, 26)
y: (64,)


In [13]:
model = SiameseLSTM(
	vocab_size=len(vocab),
	embedding_dim=settings.embed_dim,
	hidden_dim=settings.hidden_dim,
	mlp_dropout=settings.mlp_dropout,
	lstm_dropout=settings.lstm_dropout,
	embedding_matrix=embedding_matrix,
	pooling=settings.pooling,
	similarity=settings.similarity,
	mlp_layers=settings.mlp_layers,
	bidirectional=settings.bidirectional,
	concat_features=settings.concat_features,
    name=settings.siamese_name
)


In [14]:
if model.mlp:
	model.mlp.summary()


In [15]:
head_model = model.get_head_model()
head_model.summary()


Model: "siamese_head"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, None)      │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 400) │  5,479,200 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cast (Cast)         │ (None, None)      │          0 │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, None, 64)  │    119,040 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expand_dims         │ (None, None, 1)   │          0 │ cast[0][0]        │
│ (ExpandDims)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, None, 64)  │          0 │ lstm[0][0],       │
│                     │                   │            │ expand_dims[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sum_1 (Sum)         │ (None, 1)         │          0 │ expand_dims[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sum (Sum)           │ (None, 64)        │          0 │ multiply[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 1)         │          0 │ sum_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ true_divide         │ (None, 64)        │          0 │ sum[0][0],        │
│ (TrueDivide)        │                   │            │ add[0][0]         │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,598,240 (21.36 MB)

 Trainable params: 119,040 (465.00 KB)

 Non-trainable params: 5,479,200 (20.90 MB)

In [16]:
dummy_sent1 = tf.zeros((1, max_len), dtype=tf.int32)
dummy_sent2 = tf.zeros((1, max_len), dtype=tf.int32)

model((dummy_sent1, dummy_sent2))

model.summary()


Model: "lstm_mean_manhattan_noaug"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, None, 400)      │     5,479,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, None, 64)       │       119,040 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,598,240 (21.36 MB)

 Trainable params: 119,040 (465.00 KB)

 Non-trainable params: 5,479,200 (20.90 MB)

In [17]:
model = compile_model(model)

callbacks = get_callbacks(paths.siamese_path)


In [18]:
start_time = time.perf_counter()

history = model.fit(
	train_dataset,
	validation_data=dev_dataset,
	epochs=settings.epochs,
	callbacks=callbacks
)

train_time = time.perf_counter() - start_time

np.save(paths.history_path, history.history)


Epoch 1/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 52s 575ms/step - loss: 0.2507 - mae: 0.4211 - rmse: 0.5007 - val_loss: 0.1109 - val_mae: 0.2681 - val_rmse: 0.3331 - learning_rate: 0.0010
Epoch 2/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 47s 521ms/step - loss: 0.1164 - mae: 0.2839 - rmse: 0.3412 - val_loss: 0.0947 - val_mae: 0.2492 - val_rmse: 0.3078 - learning_rate: 0.0010
Epoch 3/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 48s 531ms/step - loss: 0.0946 - mae: 0.2562 - rmse: 0.3076 - val_loss: 0.0921 - val_mae: 0.2466 - val_rmse: 0.3035 - learning_rate: 0.0010
Epoch 4/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 48s 531ms/step - loss: 0.0838 - mae: 0.2399 - rmse: 0.2895 - val_loss: 0.0826 - val_mae: 0.2336 - val_rmse: 0.2874 - learning_rate: 0.0010
Epoch 5/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 48s 530ms/step - loss: 0.0763 - mae: 0.2291 - rmse: 0.2762 - val_loss: 0.0820 - val_mae: 0.2319 - val_rmse: 0.2864 - learning_rate: 0.0010
Epoch 6/20
90/90 ━━━━━━━━━━━━━━━━━━━━ 48s 528ms/step - loss: 0.0700 - mae: 0.2186 - rmse: 0.2647 - val_loss: 0.0813

In [19]:
run_config = {
    "sequence_length": max_len,
    "data_augmentation": settings.augmented_data,
    "train_time_s": train_time
}

save_json(run_config, paths.config_path)
